In [1]:
from pathlib import Path
import sys
import polars as pl
import plotly.express as px

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq3_function_lib import show_median_price_heatmap_per_region, get_borderregion_stations

# Regional price differences and price stability

In [2]:
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')
region_path = Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv')


In [3]:
year = 2022
fuel_type = "e10"

In [4]:
show_median_price_heatmap_per_region(region_price_path, region_path, year, fuel_type)

How does the price at the stations close to the german border (<=15km dist) differ from other stations in their surrounding area?

In [5]:
stations_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/stations.csv')
border_output_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations')

In [6]:
get_borderregion_stations(stations_path, border_output_path)

converting into geodata
loading border for Denmark and calculating distance.
                                            geometry  bbox_west  bbox_south  \
0  MULTIPOLYGON (((418976.869 6158111.888, 418981...   7.715325   54.451667   

   bbox_east  bbox_north   place_id  osm_type  osm_id        lat        lon  \
0  15.553064    57.95243  152522675  relation   50046  55.670249  10.333328   

      class            type  place_rank  importance addresstype     name  \
0  boundary  administrative           4    0.865532     country  Denmark   

  display_name  
0      Denmark  
loading border for Poland and calculating distance.
                                            geometry  bbox_west  bbox_south  \
0  POLYGON ((829960.088 6026277.251, 836674.578 6...  14.069639   49.002047   

   bbox_east  bbox_north   place_id  osm_type  osm_id        lat        lon  \
0  24.145783    55.03605  163056810  relation   49715  52.215933  19.134422   

      class            type  place_rank  importa

In [7]:
border_stations = pl.read_csv(border_output_path / "border_stations.csv")

print (border_stations.head())

shape: (5, 6)
┌───────────────────────────┬──────────┬───────────┬───────────────────┬───────────┬───────────────┐
│ uuid                      ┆ latitude ┆ longitude ┆ neighbour_country ┆ dist_km   ┆ border_region │
│ ---                       ┆ ---      ┆ ---       ┆ ---               ┆ ---       ┆ ---           │
│ str                       ┆ f64      ┆ f64       ┆ str               ┆ f64       ┆ str           │
╞═══════════════════════════╪══════════╪═══════════╪═══════════════════╪═══════════╪═══════════════╡
│ 005056ba-7cb6-1ed2-bceb-6 ┆ 52.55016 ┆ 13.68212  ┆ Poland            ┆ 43.342591 ┆ Surrounding   │
│ 62ba1…                    ┆          ┆           ┆                   ┆           ┆ (15-50km)     │
│ 005056ba-7cb6-1ed2-bceb-6 ┆ 51.48979 ┆ 6.78373   ┆ Netherlands       ┆ 38.94741  ┆ Surrounding   │
│ f7b23…                    ┆          ┆           ┆                   ┆           ┆ (15-50km)     │
│ 005056ba-7cb6-1ed2-bceb-9 ┆ 50.79344 ┆ 6.47057   ┆ Belgium           ┆ 22.1

In [8]:
fig = px.scatter_map(border_stations,
                    lat = "latitude",
                    lon= "longitude",
                    color = "border_region",
                    hover_name = "neighbour_country",
                    hover_data = "neighbour_country",
                    center = {"lat": 51.16, "lon": 10.45},
                    zoom = 4,
                    map_style = "open-street-map",
                    title = "Border and surrounding stations in germany")

fig.update_layout(margin = {"r":0,"t":50,"l":0,"b":0})
fig.update_traces(marker = dict(size = 15, opacity = 1))
fig.show()